# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.** My lane (Refresh / Content Opportunity Scoring) is not "is this page bad, yes/no" — it's "out of thousands of pages, which ones should an editor look at first, given they can only review a handful per week." That's a ranking question: every page gets a score, pages are sorted by that score, and the editor works down the list. The natural evaluation for this is Precision@K (how many of the top K really needed attention), not a plain accuracy number.

In [3]:
import pandas as pd

import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print("Task type: ranking/scoring over", df.shape[0], "pages, sorted by an urgency score")


Cloning into 'Capstone-Flyrank'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 143 (delta 49), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 4.86 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/Capstone-Flyrank
Rows, columns: (30000, 44)
Task type: ranking/scoring over 30000 pages, sorted by an urgency score


## 2. Target or proxy

**Proxy, not a clean target.** The label I'd use is `is_declining_label`, built from `trend_direction == "down"` (an impressions drop of 20%+ vs. the prior 30-day window). This is a proxy because it's a threshold someone chose on a current-window comparison, not an independent, already-realized future outcome. `trend_direction` and `trend_pct` are the columns that define this label, so they can never be used as model features — doing so would let the model just look up the answer instead of predicting it.

In [4]:
declining_rate = (df["trend_direction"] == "down").mean()
print(f"Proxy label positive rate (trend_direction == 'down'): {declining_rate:.1%}")
print("Columns that must be excluded from features (label source):", ["trend_direction", "trend_pct"])


Proxy label positive rate (trend_direction == 'down'): 54.2%
Columns that must be excluded from features (label source): ['trend_direction', 'trend_pct']


## 3. Success metric

**Precision@K, for K=20 and K=50.** This matches how many pages an editor can realistically review in a week, so it's a metric tied directly to the real decision, not an abstract score. "Good" means: of the top K pages by score, a high share are genuinely declining pages (per the proxy label) — higher than what the existing hand-written rule achieves.

In [5]:
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Existing hand-written rule, as a baseline reference point for "good"
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]
y = (df["trend_direction"] == "down").astype(int).values

for k in (20, 50):
    print(f"Hand-rule Precision@{k}: {precision_at_k(hand_rule_score, y, k):.3f}")


Hand-rule Precision@20: 0.900
Hand-rule Precision@50: 0.680


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), belonging to exactly one `client_id`. Below is an actual slice of the dataframe showing this.

In [6]:
unit_view = df[["content_id", "client_id", "impressions_90d", "avg_position",
                "days_since_last_update", "trend_direction"]].head(10)
print(f"One row = one content page. Total pages: {df['content_id'].nunique()}, total clients: {df['client_id'].nunique()}")
unit_view


One row = one content page. Total pages: 30000, total clients: 32


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,20,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,25,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,20,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,14,down
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,20,down
6,content_9a34b442b552,client_8722616204,20,7.0,20,down
7,content_a63219c6e95a,client_19581e27de,1724,21.2,22,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,46.0,20,down
9,content_c27558df2b0c,client_19581e27de,1240,4.9,104,down


## 5. Why ML beats a fixed rule here

We already tested this directly rather than assuming it. The current hand-written rule (`stale_visible_page`) only flags 17 pages out of 30,000 — far too narrow to be a real triage tool. But a naive small decision tree, evaluated honestly on a client-holdout split (no client's pages appearing in both train and test), scored *worse* than the hand rule at Precision@50 (0.560 vs. 0.640). So the honest case for ML here isn't "ML obviously wins" — real signals like avg_position, content_age_days, and ctr all show genuine patterns worth combining, but a careless model can also underperform a simple rule. The argument for ML is that the *signals* are real and multi-dimensional enough that combining them well should beat a rigid if-statement — but that has to be proven honestly with proper validation, not assumed.

In [7]:
# Evidence referenced above: hand rule vs. a naive tree on a client-holdout split.
# (Full comparison code lives in the Week 1 starter notebooks; summarized here as the
# justification for why this lane needs careful modeling, not just a bigger rule.)
print("Hand rule Precision@50 (client-holdout): 0.640")
print("Naive depth-2 tree Precision@50 (client-holdout): 0.560")
print("Conclusion: naive ML did not yet beat the rule -- validation matters more than model choice.")


Hand rule Precision@50 (client-holdout): 0.640
Naive depth-2 tree Precision@50 (client-holdout): 0.560
Conclusion: naive ML did not yet beat the rule -- validation matters more than model choice.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.